# 03 — Train a DQN Model Online

Train a MOUSE model directly against live `mouse-env` environments. This is the preferred example when you want the agent to keep learning from fresh interaction instead of first writing a dataset and later replaying it offline.

The run alternates two phases until `TRAIN_STEPS`:

1. **Collect** — the current model (with epsilon-greedy exploration) interacts with live envs. Each step is appended to an in-memory `Datastore`; per-env policy `contexts` feed the next action. Episode statistics for the segment just collected are printed at the end of each collect round.
2. **Train** — a `DataLoader` samples replay batches from those stores and the model is updated with `DqnObjective`.

For held-out evaluation after training, use `examples/04_inference.ipynb`. Each phase is defined below; a master cell at the bottom runs collect → train in a loop.


In [ ]:
from collections import deque

import torch

from mouse_envs import EnvConfig, make_env
from mouse_core.data import DataLoader, Datastore
from mouse_core.models import Model, push_model_to_hub
from mouse_core.models.backbone import Qwen3Backbone
from mouse_core.models.embedding import StepEmbedder
from mouse_core.models.heads import DiscreteActionValueHead
from mouse_core.objectives import DqnObjective


MODEL_ID = "mouse-example-model"
MAX_ACTIONS = 4
MAX_OBS_DISCRETE = 64
NUM_ENVS = 10 #was 1000
TRAIN_STEPS = 200000       # total env transitions (SingleEnv.step calls)
SEQUENCE_LENGTH = 512
BATCH_SIZE = 4
LEARNING_STARTS = 20000    # env transitions before first gradient update
EXPLORATION_FULL_AT_STEP = 10000
COLLECT_STEPS = 10        # consecutive env transitions per env before moving to the next
GRAD_STEPS = 10           # gradient updates per learn phase


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


Online training keeps the environment in the training loop. Each `SingleEnv` instance runs one infinite task with deterministic FrozenLake dynamics and 50-step episode truncation, so the model keeps carrying policy context across episode boundaries.


In [2]:
configs = [
    EnvConfig(
        id="Procedural-FrozenLake-v1",
        name=f"proc_frozenlake_online_{i}",
        reset_seed=i,
        episodes_per_task=0,
        kwargs={
            "is_slippery": False,
            "min_width": 4, "max_width": 8,
            "min_height": 4, "max_height": 8,
            "step_penalty": -0.01,
            "max_episode_steps": 50,
            "map_seed": i,
        },
    )
    for i in range(NUM_ENVS)
]

envs = [make_env(config) for config in configs]
print(f"Environments {min(10, len(envs))} of {len(envs)}:")
for env in envs[:10]:
    print(env.name)

Environments 10 of 1000:
proc_frozenlake_online_0
proc_frozenlake_online_1
proc_frozenlake_online_2
proc_frozenlake_online_3
proc_frozenlake_online_4
proc_frozenlake_online_5
proc_frozenlake_online_6
proc_frozenlake_online_7
proc_frozenlake_online_8
proc_frozenlake_online_9


## Build the model

The model uses a `StepEmbedder`, the full pretrained `Qwen3Backbone`, and a `DiscreteActionValueHead`.

For faster debugging you can pass `num_layers=...` to truncate the backbone.

In [3]:
backbone = Qwen3Backbone(pretrained="Qwen/Qwen3-0.6B")

encoder = StepEmbedder(
    hidden_dim=backbone.hidden_dim,
    modalities=[
        {
            "field": "action",
            "type": "discrete",
            "vocab_size": MAX_ACTIONS,
            "std": 0.02,
            "tokens": 1,
        },
        {
            "field": "observation",
            "type": "discrete",
            "vocab_size": MAX_OBS_DISCRETE,
            "std": 0.02,
            "tokens": 1,
        },
        {
            "field": "reward",
            "type": "rff",
            "std": 0.02,
            "in_min": 0.01,
            "in_max": 100.0,
            "tokens": 1,
        },
        {
            "field": "done",
            "type": "discrete",
            "vocab_size": 5,
            "std": 0.02,
            "tokens": 1,
        },
    ],
    modality_fusion="sum",
    include_type_token=False,
)

head = DiscreteActionValueHead(
    in_features=backbone.hidden_dim,
    out_features=MAX_ACTIONS,
    hidden_dim=backbone.hidden_dim,
    num_layers=1,
    scale=0.1,
)

model = Model(encoder=encoder, backbone=backbone, heads=head).train().to(device)
print(model)

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Model(
  (encoder): StepEmbedder(
    (type_embedder): TypeEmbedder(
      (embed): ScaledEmbedding(64, 1024)
    )
    (modality_embedders): ModuleDict(
      (action): DiscreteEmbedder(
        (embed): ScaledEmbedding(4, 1024)
      )
      (observation): DiscreteEmbedder(
        (embed): ScaledEmbedding(64, 1024)
      )
      (reward): ScalarRFFEmbedder(
        (rff): RandomFourierFeatures()
      )
      (done): DiscreteEmbedder(
        (embed): ScaledEmbedding(5, 1024)
      )
    )
  )
  (backbone): Qwen3Backbone(
    (model): Qwen3Model(
      (embed_tokens): Embedding(1, 1024)
      (layers): ModuleList(
        (0-27): 28 x Qwen3DecoderLayer(
          (self_attn): Qwen3Attention(
            (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
            (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
            (o_proj): Linear(in_features=2048, out_features=10

## Replay buffer and policy contexts

Each env writes to one `Datastore`; together they are the live replay buffer. Each env also has a `contexts` deque capped at `SEQUENCE_LENGTH` — the action-selection history used during collection. Contexts are never cleared; episode boundaries appear as `done` values in the sequence.


In [4]:
stores = [Datastore(name=env.name) for env in envs]
contexts = [deque(maxlen=SEQUENCE_LENGTH) for _ in envs]


## Collection phase

One **collect round** visits every env once. For each env the loop runs up to `COLLECT_STEPS` consecutive transitions with a single **B=1 KV cache** (like vLLM per-sequence caching): full-context pass on the first model call, then incremental passes for new rows only. The cache is discarded when moving to the next env — inference is never batched across envs, so only one cache is needed at a time.

1. Choose actions with epsilon-greedy exploration. Random actions skip the model; the next model call feeds all uncached rows as a catch-up slice.
2. Step the env, append to its `Datastore` and `contexts` deque. The KV cache grows incrementally for the duration of that env's visit and is discarded when the loop moves to the next env.

At the end of each collect round, `print_collect_stats` reports episodes completed during that segment (first 10 envs).


In [5]:
def epsilon_for_step(step: int) -> float:
    if not 0 < EXPLORATION_FULL_AT_STEP < TRAIN_STEPS:
        raise ValueError("EXPLORATION_FULL_AT_STEP must be in [1, TRAIN_STEPS).")
    return min(step / EXPLORATION_FULL_AT_STEP, 1.0)


def print_collect_stats(
    envs,
    stores: list[Datastore],
    episodes_before: list[int],
    *,
    total_steps: int,
    preview: int = 10,
) -> None:
    """Print episode stats for episodes completed during the last collect round."""
    replay_steps = sum(len(store) for store in stores)
    epsilon = epsilon_for_step(total_steps)
    print(
        f"step={total_steps} epsilon={epsilon:.3f} replay_steps={replay_steps} "
        f"({min(preview, len(envs))} of {len(envs)} envs):"
    )
    for i, env in enumerate(envs[:preview]):
        rewards = env.tracker.episode_cum_rewards[episodes_before[i]:]
        lengths = env.tracker.episode_lengths[episodes_before[i]:]
        n = len(rewards)
        mean_r = torch.tensor(rewards).mean().item() if n else float("nan")
        mean_l = torch.tensor(lengths).float().mean().item() if n else float("nan")
        print(f"  {env.name}: +{n} episodes | mean reward {mean_r:.3f} | mean length {mean_l:.1f}")


In [6]:
def collect_round(
    model: Model,
    envs: list,
    stores: list[Datastore],
    contexts: list[deque],
    total_steps: int,
) -> int:
    """Run one collect round over all envs. Returns the updated step count."""
    episodes_before = [len(env.tracker.episode_cum_rewards) for env in envs]
    model.eval()
    for env, store, context in zip(envs, stores, contexts):
        if total_steps >= TRAIN_STEPS:
            break

        kv_cache = None
        cache_count = 0
        ctx_count = 0
        collect_n = min(COLLECT_STEPS, TRAIN_STEPS - total_steps)

        for _ in range(collect_n):
            epsilon = epsilon_for_step(total_steps)

            if not context or torch.rand(1).item() < epsilon:
                inp = env.sample_random_input()
            else:
                ctx_list = list(context)
                with torch.no_grad():
                    if kv_cache is None:
                        preds, _, kv_cache = model([ctx_list], use_cache=True)
                    else:
                        uncached = ctx_count - cache_count
                        preds, _, kv_cache = model(
                            [ctx_list[-uncached:]], cache=kv_cache, use_cache=True
                        )
                    cache_count = ctx_count

                action = model.get_action(preds, temperature=0.0, num_actions=MAX_ACTIONS)
                inp = {"action": action.squeeze().cpu()}

            out = env.step(inp)
            store.append({**inp, **out})
            context.append({**inp, **out})
            ctx_count += 1
            total_steps += 1

    print_collect_stats(envs, stores, episodes_before, total_steps=total_steps)
    return total_steps


## Training phase

One **train round** runs after each collect round once `total_steps >= LEARNING_STARTS`:

1. Call `loader.refresh()` so the replay sampler re-snapshots stores and drops prefetched batches.
2. Sample `GRAD_STEPS` batches and run the DQN update on each.

The loader is created up front (stores may still be empty; with `pack=True`, sampling starts once rows have been appended). It uses `pack=True`, so stores shorter than `SEQUENCE_LENGTH` are padded by drawing additional segments. `weight_mode="per_step"` weights sampling toward larger stores. Pass `seed=...` for reproducible batches; default is non-deterministic.


In [7]:
loader = DataLoader(
    stores,
    sequence_length=SEQUENCE_LENGTH,
    batch_size=BATCH_SIZE,
    weight_mode="per_step",
    pack=True,
    num_workers=0,
)
objective = DqnObjective(
    gamma_step=1.0,
    gamma_episode_terminal=0.99,
    gamma_episode_truncated=0.99,
    gamma_task_terminal=0.99,
    gamma_task_truncated=0.99,
    tau=0.0005,
)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1.0e-5,
    weight_decay=0.0,
    betas=(0.9, 0.95),
    eps=1.0e-8,
)

In [8]:
def train_round(
    model: Model,
    optimizer: torch.optim.Optimizer,
    objective: DqnObjective,
    loader: DataLoader,
) -> tuple[torch.Tensor, dict[str, float]]:
    """Refresh replay and run ``GRAD_STEPS`` gradient updates."""
    model.train()
    loader.refresh()

    loss: torch.Tensor | None = None
    metrics: dict[str, float] = {}
    for _ in range(GRAD_STEPS):
        batch = loader.next_batch()
        
        predictions, objective_data, _ = model(batch)
        loss, metrics = objective(objective_data, predictions)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        model.polyak_update(action_value_tau=objective.tau)

    assert loss is not None
    return loss, metrics


## Run online training

The master loop alternates **collect → train** until `TRAIN_STEPS` env transitions have been taken. Training is skipped until `LEARNING_STARTS`. Each collect round prints segment episode stats; training progress is logged every 1000 env transitions.


In [ ]:
total_steps = 0
loss, metrics = None, {}

while total_steps < TRAIN_STEPS:
    total_steps = collect_round(model, envs, stores, contexts, total_steps)

    if total_steps >= LEARNING_STARTS:
        loss, metrics = train_round(model, optimizer, objective, loader)

        if total_steps % 1000 == 0 and loss is not None:
            print(
                f"train loss={loss.item():.4f} q={metrics['q_values_mean']:.3f}"
            )

loader.close()

for env in envs:
    env.close()
print("Online training finished.")


: 

## Push to the Hub

Run this cell if you want to save the online-trained checkpoint to the shared Hub repo under `MODEL_ID`. The inference notebook loads the current checkpoint from that repo, so whichever training notebook you push last is the model it evaluates.

In [ ]:
model.eval().to("cpu")
url = push_model_to_hub(model, MODEL_ID, private=False, clear=True)
print(f"Pushed to {url}")